In [1]:
!pip -q install numba numpy

In [2]:
import numpy as np
from numba import cuda

In [3]:
@cuda.jit
def vector_add_1d(a, b, out):
    i = cuda.grid(1)

    if i < a.size:
        out[i] = a[i] + b[i]

A = np.array([1, 2, 3, 4], dtype=np.float32)
B = np.array([5, 6, 7, 8], dtype=np.float32)

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array_like(A)

threads = 256
blocks = (A.size + threads - 1) // threads

vector_add_1d[blocks, threads](d_A, d_B, d_C)

C = d_C.copy_to_host()

print("A =", A)
print("B =", B)
print("C =", C)

np.testing.assert_allclose(C, A + B, atol=1e-5)
print("Task 1: PASS")

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


A = [1. 2. 3. 4.]
B = [5. 6. 7. 8.]
C = [ 6.  8. 10. 12.]
Task 1: PASS


In [4]:
@cuda.jit
def vector_scale_grid_stride(x, alpha, out):
    i = cuda.grid(1)
    stride = cuda.gridsize(1)

    while i < x.size:
        out[i] = x[i] * alpha
        i += stride

X = np.array([1, 2, 3, 4, 5], dtype=np.float32)
alpha = np.float32(3.0)

d_X = cuda.to_device(X)
d_Y = cuda.device_array_like(X)

vector_scale_grid_stride[64, 256](d_X, alpha, d_Y)

Y = d_Y.copy_to_host()

print("X =", X)
print("Alpha =", alpha)
print("Y =", Y)

np.testing.assert_allclose(Y, X * alpha, atol=1e-5)
print("Task 2: PASS")

X = [1. 2. 3. 4. 5.]
Alpha = 3.0
Y = [ 3.  6.  9. 12. 15.]
Task 2: PASS


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 64 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [5]:
@cuda.jit
def matrix_add_2d(A, B, C):
    col, row = cuda.grid(2)

    if row < A.shape[0] and col < A.shape[1]:
        C[row, col] = A[row, col] + B[row, col]

A = np.array([[1, 2], [3, 4]], dtype=np.float32)
B = np.array([[5, 6], [7, 8]], dtype=np.float32)

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array_like(A)

threads = (16, 16)
blocks = (
    (A.shape[1] + 15) // 16,
    (A.shape[0] + 15) // 16
)

matrix_add_2d[blocks, threads](d_A, d_B, d_C)

C = d_C.copy_to_host()

print("Matrix A:\n", A)
print("Matrix B:\n", B)
print("Result C:\n", C)

np.testing.assert_allclose(C, A + B, atol=1e-5)
print("Task 3: PASS")

Matrix A:
 [[1. 2.]
 [3. 4.]]
Matrix B:
 [[5. 6.]
 [7. 8.]]
Result C:
 [[ 6.  8.]
 [10. 12.]]
Task 3: PASS


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [6]:
@cuda.jit
def matrix_transpose_naive(A, B):
    col, row = cuda.grid(2)

    if row < A.shape[0] and col < A.shape[1]:
        B[col, row] = A[row, col]

A = np.array([
    [1, 2, 3],
    [4, 5, 6]
], dtype=np.float32)

d_A = cuda.to_device(A)
d_B = cuda.device_array(
    (A.shape[1], A.shape[0]),
    dtype=np.float32
)

threads = (16, 16)
blocks = (
    (A.shape[1] + 15) // 16,
    (A.shape[0] + 15) // 16
)

matrix_transpose_naive[blocks, threads](d_A, d_B)

B = d_B.copy_to_host()

print("Original matrix:\n", A)
print("Transposed matrix:\n", B)

np.testing.assert_allclose(B, A.T, atol=1e-5)
print("Task 4: PASS")

Original matrix:
 [[1. 2. 3.]
 [4. 5. 6.]]
Transposed matrix:
 [[1. 4.]
 [2. 5.]
 [3. 6.]]
Task 4: PASS


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [7]:
@cuda.jit
def volume_init_3d(vol, D, H, W):
    x, y, z = cuda.grid(3)

    if x < W and y < H and z < D:
        vol[z, y, x] = z * 10000 + y * 100 + x

D, H, W = 3, 4, 5

d_vol = cuda.device_array(
    (D, H, W),
    dtype=np.int32
)

threads = (8, 8, 8)
blocks = (
    (W + 7) // 8,
    (H + 7) // 8,
    (D + 7) // 8
)

volume_init_3d[blocks, threads](d_vol, D, H, W)

volume = d_vol.copy_to_host()

print("3D Volume:\n", volume)
print("Volume shape:", volume.shape)

expected = np.fromfunction(
    lambda z, y, x: z * 10000 + y * 100 + x,
    (D, H, W),
    dtype=int
)

np.testing.assert_array_equal(volume, expected)
print("Task 5: PASS")

3D Volume:
 [[[    0     1     2     3     4]
  [  100   101   102   103   104]
  [  200   201   202   203   204]
  [  300   301   302   303   304]]

 [[10000 10001 10002 10003 10004]
  [10100 10101 10102 10103 10104]
  [10200 10201 10202 10203 10204]
  [10300 10301 10302 10303 10304]]

 [[20000 20001 20002 20003 20004]
  [20100 20101 20102 20103 20104]
  [20200 20201 20202 20203 20204]
  [20300 20301 20302 20303 20304]]]
Volume shape: (3, 4, 5)
Task 5: PASS


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [8]:
@cuda.jit
def rgb_to_grayscale(img_rgb, img_gray):
    x, y = cuda.grid(2)

    if y < img_rgb.shape[0] and x < img_rgb.shape[1]:
        R = img_rgb[y, x, 0]
        G = img_rgb[y, x, 1]
        B = img_rgb[y, x, 2]

        img_gray[y, x] = (
            0.299 * R +
            0.587 * G +
            0.114 * B
        )

img_rgb = np.array([
    [[255, 0, 0], [0, 255, 0]],
    [[0, 0, 255], [255, 255, 255]]
], dtype=np.float32)

d_rgb = cuda.to_device(img_rgb)

d_gray = cuda.device_array(
    (img_rgb.shape[0], img_rgb.shape[1]),
    dtype=np.float32
)

threads = (16, 16)
blocks = (1, 1)

rgb_to_grayscale[blocks, threads](d_rgb, d_gray)

gray = d_gray.copy_to_host()

print("Original RGB:\n", img_rgb)
print("Grayscale:\n", gray)

expected = (
    0.299 * img_rgb[:, :, 0] +
    0.587 * img_rgb[:, :, 1] +
    0.114 * img_rgb[:, :, 2]
)

np.testing.assert_allclose(gray, expected, atol=1e-4)
print("Task 6: PASS")

Original RGB:
 [[[255.   0.   0.]
  [  0. 255.   0.]]

 [[  0.   0. 255.]
  [255. 255. 255.]]]
Grayscale:
 [[ 76.245 149.685]
 [ 29.07  255.   ]]
Task 6: PASS


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [9]:
@cuda.jit
def vector_lerp(a, b, t, out):
    i = cuda.grid(1)

    if i < a.size:
        out[i] = (1.0 - t[i]) * a[i] + t[i] * b[i]

A = np.array([10, 20, 30, 40], dtype=np.float32)
B = np.array([20, 40, 60, 80], dtype=np.float32)

T = np.array([0.0, 0.25, 0.5, 1.0], dtype=np.float32)

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_T = cuda.to_device(T)
d_out = cuda.device_array_like(A)

threads = 256
blocks = (A.size + threads - 1) // threads

vector_lerp[blocks, threads](d_A, d_B, d_T, d_out)

result = d_out.copy_to_host()

print("A =", A)
print("B =", B)
print("T =", T)
print("LERP result =", result)

expected = (1 - T) * A + T * B

np.testing.assert_allclose(result, expected, atol=1e-5)
print("Task 7: PASS")

A = [10. 20. 30. 40.]
B = [20. 40. 60. 80.]
T = [0.   0.25 0.5  1.  ]
LERP result = [10. 25. 45. 80.]
Task 7: PASS


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [10]:
import numpy as np
from numba import cuda

@cuda.jit
def vector_threshold(x, thresh, mask):
    i = cuda.grid(1)
    stride = cuda.gridsize(1)

    while i < x.size:
        mask[i] = 1 if x[i] >= thresh else 0
        i += stride

X = np.array([2, 7, 4, 9, 5], dtype=np.float32)
threshold = 5.0

d_x = cuda.to_device(X)
d_mask = cuda.device_array(X.size, dtype=np.uint8)

vector_threshold[1, 256](d_x, threshold, d_mask)

result = d_mask.copy_to_host()

print("Input:", X)
print("Threshold:", threshold)
print("Mask:", result)

np.testing.assert_array_equal(
    result, (X >= threshold).astype(np.uint8)
)
print("Task 8: PASS")

Input: [2. 7. 4. 9. 5.]
Threshold: 5.0
Mask: [0 1 0 1 1]
Task 8: PASS


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [11]:
import time

@cuda.jit
def benchmark_strided_access(in_arr, out_arr, stride):
    i = cuda.grid(1)

    if i < out_arr.size:
        out_arr[i] = in_arr[i * stride]

N = 1_000_000
threads = 256
blocks = (N + threads - 1) // threads

for stride in [1, 2, 4, 8, 16, 32]:
    data = np.arange(N * stride, dtype=np.float32)

    d_input = cuda.to_device(data)
    d_output = cuda.device_array(N, dtype=np.float32)

    # Compile and warm up
    benchmark_strided_access[blocks, threads](
        d_input, d_output, stride
    )
    cuda.synchronize()

    start = time.perf_counter()

    for _ in range(20):
        benchmark_strided_access[blocks, threads](
            d_input, d_output, stride
        )

    cuda.synchronize()
    elapsed = (time.perf_counter() - start) / 20

    result = d_output.copy_to_host()

    np.testing.assert_array_equal(
        result, data[::stride]
    )

    print(f"Stride {stride:2d}: {elapsed * 1000:.3f} ms")

print("Task 9: PASS")

Stride  1: 0.054 ms
Stride  2: 0.053 ms
Stride  4: 0.088 ms
Stride  8: 0.141 ms
Stride 16: 0.355 ms
Stride 32: 0.306 ms
Task 9: PASS


In [13]:
import numpy as np
from numba import cuda, float32

@cuda.jit
def stencil_1d_shared(in_arr, out_arr, radius):

    # Shared memory: 256 + 2 halo elements
    shared = cuda.shared.array(
        shape=258,
        dtype=float32
    )

    tx = cuda.threadIdx.x
    i = cuda.blockIdx.x * cuda.blockDim.x + tx
    n = in_arr.size

    # Load center element
    if i < n:
        shared[tx + 1] = in_arr[i]
    else:
        shared[tx + 1] = in_arr[n - 1]

    # Load left halo
    if tx == 0:
        left = i - 1
        if left < 0:
            left = 0
        shared[0] = in_arr[left]

    # Load right halo
    if tx == 255:
        right = i + 1
        if right >= n:
            right = n - 1
        shared[257] = in_arr[right]

    cuda.syncthreads()

    # Moving average
    if i < n:
        total = (
            shared[tx] +
            shared[tx + 1] +
            shared[tx + 2]
        )

        out_arr[i] = total / 3.0


# Test data
X = np.array([1, 2, 3, 4, 5], dtype=np.float32)

d_x = cuda.to_device(X)
d_y = cuda.device_array_like(X)

threads = 256
blocks = (X.size + threads - 1) // threads

stencil_1d_shared[blocks, threads](
    d_x, d_y, 1
)

result = d_y.copy_to_host()

expected = np.array([
    1.3333333,
    2.0,
    3.0,
    4.0,
    4.6666667
], dtype=np.float32)

print("Input:", X)
print("Moving Average:", result)

np.testing.assert_allclose(
    result,
    expected,
    atol=1e-4
)

print("Task 10: PASS")

Input: [1. 2. 3. 4. 5.]
Moving Average: [1.3333334 2.        3.        4.        4.6666665]
Task 10: PASS


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [14]:
import numpy as np
from numba import cuda, float32

@cuda.jit
def conv_1d_shared_tiled(x, kernel, y):

    # 256 threads + 2 halo elements
    shared = cuda.shared.array(
        shape=258,
        dtype=float32
    )

    tx = cuda.threadIdx.x
    i = cuda.blockIdx.x * cuda.blockDim.x + tx
    n = x.size

    # Load center values
    if i < n:
        shared[tx + 1] = x[i]
    else:
        shared[tx + 1] = 0.0

    # Left halo
    if tx == 0:
        if i > 0:
            shared[0] = x[i - 1]
        else:
            shared[0] = 0.0

    # Right halo
    if tx == 255:
        if i + 1 < n:
            shared[257] = x[i + 1]
        else:
            shared[257] = 0.0

    cuda.syncthreads()

    # Convolution
    if i < n:
        result = 0.0

        for j in range(3):
            result += shared[tx + j] * kernel[2 - j]

        y[i] = result


# Input data
X = np.array([1, 2, 3, 4, 5], dtype=np.float32)

K = np.array(
    [0.25, 0.5, 0.25],
    dtype=np.float32
)

# Copy to GPU
d_x = cuda.to_device(X)
d_k = cuda.to_device(K)
d_y = cuda.device_array_like(X)

threads = 256
blocks = (X.size + threads - 1) // threads

# Run CUDA kernel
conv_1d_shared_tiled[blocks, threads](
    d_x, d_k, d_y
)

# Copy result to CPU
result = d_y.copy_to_host()

# CPU verification
expected = np.convolve(X, K, mode="same")

print("Input:", X)
print("Kernel:", K)
print("GPU Result:", result)
print("CPU Result:", expected)

np.testing.assert_allclose(
    result,
    expected,
    atol=1e-4
)

print("Task 11: PASS")

Input: [1. 2. 3. 4. 5.]
Kernel: [0.25 0.5  0.25]
GPU Result: [1.  2.  3.  4.  3.5]
CPU Result: [1.  2.  3.  4.  3.5]
Task 11: PASS


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [15]:
@cuda.jit
def matmul_tiled_shared(A, B, C):
    sA = cuda.shared.array((16, 16), dtype=float32)
    sB = cuda.shared.array((16, 16), dtype=float32)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    row = cuda.blockIdx.y * 16 + ty
    col = cuda.blockIdx.x * 16 + tx

    total = 0.0

    for phase in range((A.shape[1] + 15) // 16):
        a_col = phase * 16 + tx
        b_row = phase * 16 + ty

        if row < A.shape[0] and a_col < A.shape[1]:
            sA[ty, tx] = A[row, a_col]
        else:
            sA[ty, tx] = 0.0

        if b_row < B.shape[0] and col < B.shape[1]:
            sB[ty, tx] = B[b_row, col]
        else:
            sB[ty, tx] = 0.0

        cuda.syncthreads()

        for k in range(16):
            total += sA[ty, k] * sB[k, tx]

        cuda.syncthreads()

    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = total

A = np.array([[1, 2], [3, 4]], dtype=np.float32)
B = np.array([[5, 6], [7, 8]], dtype=np.float32)

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array((2, 2), dtype=np.float32)

matmul_tiled_shared[(1, 1), (16, 16)](
    d_A, d_B, d_C
)

C = d_C.copy_to_host()

print("A:\n", A)
print("B:\n", B)
print("A x B:\n", C)

np.testing.assert_allclose(C, A @ B, atol=1e-3)
print("Task 12: PASS")

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


A:
 [[1. 2.]
 [3. 4.]]
B:
 [[5. 6.]
 [7. 8.]]
A x B:
 [[19. 22.]
 [43. 50.]]
Task 12: PASS


In [16]:
@cuda.jit
def transpose_bank_conflict_free(A, B):
    tile = cuda.shared.array((32, 33), dtype=float32)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    x = cuda.blockIdx.x * 32 + tx
    y = cuda.blockIdx.y * 32 + ty

    if y < A.shape[0] and x < A.shape[1]:
        tile[ty, tx] = A[y, x]

    cuda.syncthreads()

    x_out = cuda.blockIdx.y * 32 + tx
    y_out = cuda.blockIdx.x * 32 + ty

    if y_out < B.shape[0] and x_out < B.shape[1]:
        B[y_out, x_out] = tile[tx, ty]

A = np.arange(64 * 64, dtype=np.float32).reshape(64, 64)

d_A = cuda.to_device(A)
d_B = cuda.device_array((64, 64), dtype=np.float32)

transpose_bank_conflict_free[(2, 2), (32, 32)](
    d_A, d_B
)

B = d_B.copy_to_host()

np.testing.assert_allclose(B, A.T, atol=1e-5)

print("Original shape:", A.shape)
print("Transposed shape:", B.shape)
print("Task 13: PASS")

Original shape: (64, 64)
Transposed shape: (64, 64)
Task 13: PASS


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [17]:
@cuda.jit
def reduction_sum_shared(in_arr, block_sums):
    shared = cuda.shared.array(256, dtype=float32)

    tx = cuda.threadIdx.x
    i = cuda.blockIdx.x * cuda.blockDim.x + tx

    if i < in_arr.size:
        shared[tx] = in_arr[i]
    else:
        shared[tx] = 0.0

    cuda.syncthreads()

    stride = 128

    while stride > 0:
        if tx < stride:
            shared[tx] += shared[tx + stride]

        cuda.syncthreads()
        stride //= 2

    if tx == 0:
        block_sums[cuda.blockIdx.x] = shared[0]

X = np.arange(1, 1001, dtype=np.float32)

d_x = cuda.to_device(X)

threads = 256
blocks = (X.size + threads - 1) // threads

d_sums = cuda.device_array(blocks, dtype=np.float32)

reduction_sum_shared[blocks, threads](
    d_x, d_sums
)

block_sums = d_sums.copy_to_host()

gpu_sum = np.sum(block_sums)
cpu_sum = np.sum(X)

print("GPU sum:", gpu_sum)
print("CPU sum:", cpu_sum)

np.testing.assert_allclose(
    gpu_sum, cpu_sum, rtol=1e-4
)
print("Task 14: PASS")

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


GPU sum: 500500.0
CPU sum: 500500.0
Task 14: PASS


In [18]:
import numpy as np
from numba import cuda, float32, int64

@cuda.jit
def reduction_argmax_shared(in_arr, block_max_vals,
                            block_max_indices):

    s_vals = cuda.shared.array(256, dtype=float32)
    s_idx = cuda.shared.array(256, dtype=int64)

    tx = cuda.threadIdx.x
    i = cuda.blockIdx.x * 256 + tx

    if i < in_arr.size:
        s_vals[tx] = in_arr[i]
        s_idx[tx] = i
    else:
        s_vals[tx] = -float("inf")
        s_idx[tx] = -1

    cuda.syncthreads()

    stride = 128

    while stride > 0:
        if tx < stride:
            v1 = s_vals[tx]
            v2 = s_vals[tx + stride]
            idx1 = s_idx[tx]
            idx2 = s_idx[tx + stride]

            if v2 > v1 or (v2 == v1 and idx2 >= 0
                           and (idx1 < 0 or idx2 < idx1)):
                s_vals[tx] = v2
                s_idx[tx] = idx2

        cuda.syncthreads()
        stride //= 2

    if tx == 0:
        block_max_vals[cuda.blockIdx.x] = s_vals[0]
        block_max_indices[cuda.blockIdx.x] = s_idx[0]


X = np.array([3, 8, 2, 10, 5, 7, 10, 1],
             dtype=np.float32)

d_x = cuda.to_device(X)

blocks = (X.size + 255) // 256

d_vals = cuda.device_array(blocks, dtype=np.float32)
d_idx = cuda.device_array(blocks, dtype=np.int64)

reduction_argmax_shared[blocks, 256](
    d_x, d_vals, d_idx
)

vals = d_vals.copy_to_host()
indices = d_idx.copy_to_host()

best = np.argmax(vals)
gpu_max = vals[best]
gpu_index = indices[best]

print("Input:", X)
print("Maximum:", gpu_max)
print("Index:", gpu_index)

assert gpu_max == np.max(X)
assert gpu_index == np.argmax(X)

print("Task 15: PASS")

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Input: [ 3.  8.  2. 10.  5.  7. 10.  1.]
Maximum: 10.0
Index: 3
Task 15: PASS


In [19]:
import numpy as np
from numba import cuda

# 64 filter coefficients
weights = np.ones(64, dtype=np.float32) / 64

@cuda.jit
def constant_memory_filter(signal, out):

    w = cuda.const.array_like(weights)

    i = cuda.grid(1)
    n = signal.size

    if i < n:
        total = 0.0

        for k in range(64):
            idx = i + k - 32

            if 0 <= idx < n:
                total += signal[idx] * w[k]

        out[i] = total


X = np.arange(1, 101, dtype=np.float32)

d_x = cuda.to_device(X)
d_y = cuda.device_array_like(X)

threads = 256
blocks = (X.size + 255) // 256

constant_memory_filter[blocks, threads](d_x, d_y)

result = d_y.copy_to_host()

expected = np.convolve(X, weights, mode="same")

print("GPU:", result[:10])
print("CPU:", expected[:10])

np.testing.assert_allclose(
    result, expected, atol=1e-4
)

print("Task 16: PASS")

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


GPU: [ 8.25      8.765625  9.296875  9.84375  10.40625  10.984375 11.578125
 12.1875   12.8125   13.453125]
CPU: [ 8.25      8.765625  9.296875  9.84375  10.40625  10.984375 11.578125
 12.1875   12.8125   13.453125]
Task 16: PASS


In [20]:
import numpy as np
from numba import cuda

# 64 filter coefficients
weights = np.ones(64, dtype=np.float32) / 64

@cuda.jit
def constant_memory_filter(signal, out):

    w = cuda.const.array_like(weights)

    i = cuda.grid(1)
    n = signal.size

    if i < n:
        total = 0.0

        for k in range(64):
            idx = i + k - 32

            if 0 <= idx < n:
                total += signal[idx] * w[k]

        out[i] = total


X = np.arange(1, 101, dtype=np.float32)

d_x = cuda.to_device(X)
d_y = cuda.device_array_like(X)

threads = 256
blocks = (X.size + 255) // 256

constant_memory_filter[blocks, threads](d_x, d_y)

result = d_y.copy_to_host()

expected = np.convolve(X, weights, mode="same")

print("GPU:", result[:10])
print("CPU:", expected[:10])

np.testing.assert_allclose(
    result, expected, atol=1e-4
)

print("Task 16: PASS")

GPU: [ 8.25      8.765625  9.296875  9.84375  10.40625  10.984375 11.578125
 12.1875   12.8125   13.453125]
CPU: [ 8.25      8.765625  9.296875  9.84375  10.40625  10.984375 11.578125
 12.1875   12.8125   13.453125]
Task 16: PASS


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [21]:
import numpy as np
from numba import cuda
import time

def benchmark_pinned_transfer(size_bytes):

    n = size_bytes // 4

    pageable = np.ones(n, dtype=np.float32)

    pinned = cuda.pinned_array(
        n, dtype=np.float32
    )
    pinned[:] = pageable

    d_arr = cuda.device_array(n, dtype=np.float32)

    def measure(arr, repeats=20):
        times = []

        for _ in range(repeats):
            start = time.perf_counter()

            d_arr.copy_to_device(arr)
            cuda.synchronize()

            elapsed = time.perf_counter() - start
            times.append(elapsed)

        return np.median(times)

    # Warm-up
    d_arr.copy_to_device(pageable)
    cuda.synchronize()

    t_pageable = measure(pageable)
    t_pinned = measure(pinned)

    gb = size_bytes / 1e9

    bw_pageable = gb / t_pageable
    bw_pinned = gb / t_pinned

    print("Pageable bandwidth:",
          round(bw_pageable, 3), "GB/s")

    print("Pinned bandwidth:",
          round(bw_pinned, 3), "GB/s")

    print("Speedup:",
          round(bw_pinned / bw_pageable, 2), "x")

    np.testing.assert_array_equal(
        d_arr.copy_to_host(), pageable
    )

    print("Task 17: PASS (data integrity)")


benchmark_pinned_transfer(64 * 1024 * 1024)

Pageable bandwidth: 2.092 GB/s
Pinned bandwidth: 11.422 GB/s
Speedup: 5.46 x
Task 17: PASS (data integrity)


In [22]:
import numpy as np
from numba import cuda

@cuda.jit
def warp_reduce_sum_shfl(in_arr, out_arr):

    i = cuda.grid(1)
    lane = cuda.threadIdx.x % 32
    warp_id = i // 32

    value = 0.0

    if i < in_arr.size:
        value = in_arr[i]

    mask = 0xFFFFFFFF

    for offset in (16, 8, 4, 2, 1):
        value += cuda.shfl_down_sync(
            mask, value, offset
        )

    if lane == 0:
        out_arr[warp_id] = value


X = np.arange(1, 1025, dtype=np.float32)

d_x = cuda.to_device(X)

num_warps = (X.size + 31) // 32
d_out = cuda.device_array(
    num_warps, dtype=np.float32
)

threads = 256
blocks = (X.size + threads - 1) // threads

warp_reduce_sum_shfl[blocks, threads](
    d_x, d_out
)

result = d_out.copy_to_host()

gpu_sum = np.sum(result)
cpu_sum = np.sum(X)

print("GPU sum:", gpu_sum)
print("CPU sum:", cpu_sum)

np.testing.assert_allclose(
    gpu_sum, cpu_sum, rtol=1e-4
)

print("Task 18: PASS")

GPU sum: 524800.0
CPU sum: 524800.0
Task 18: PASS


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [23]:
import numpy as np
from numba import cuda

@cuda.jit
def warp_vote_analyzer(data, mask_out):

    i = cuda.grid(1)
    lane = cuda.threadIdx.x % 32
    warp_id = i // 32

    active = i < data.size
    predicate = False

    if active:
        predicate = data[i] > 0

    mask = 0xFFFFFFFF

    ballot = cuda.ballot_sync(mask, predicate)
    all_true = cuda.all_sync(mask, predicate)
    any_true = cuda.any_sync(mask, predicate)

    if lane == 0:
        mask_out[warp_id, 0] = ballot
        mask_out[warp_id, 1] = all_true
        mask_out[warp_id, 2] = any_true


# Three full warps
X = np.concatenate([
    np.ones(32, dtype=np.float32),
    -np.ones(32, dtype=np.float32),
    np.array([1, -1] * 16, dtype=np.float32)
])

d_x = cuda.to_device(X)

d_out = cuda.device_array(
    (3, 3), dtype=np.int64
)

warp_vote_analyzer[1, 96](d_x, d_out)

result = d_out.copy_to_host()

print("Ballot / All / Any:")
print(result)

assert result[0, 1] == 1
assert result[1, 2] == 0
assert result[2, 2] == 1
assert result[2, 1] == 0

print("Task 19: PASS")

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Ballot / All / Any:
[[        -1          1          1]
 [         0          0          0]
 [1431655765          0          1]]
Task 19: PASS


In [24]:
import numpy as np
from numba import cuda, float32

@cuda.jit
def scan_hillis_steele(x, out):

    s1 = cuda.shared.array(1024, dtype=float32)
    s2 = cuda.shared.array(1024, dtype=float32)

    tx = cuda.threadIdx.x
    n = x.size

    if tx < n:
        s1[tx] = x[tx]
    else:
        s1[tx] = 0.0

    cuda.syncthreads()

    stride = 1

    while stride < cuda.blockDim.x:

        if tx >= stride:
            s2[tx] = s1[tx] + s1[tx - stride]
        else:
            s2[tx] = s1[tx]

        cuda.syncthreads()

        s1[tx] = s2[tx]

        cuda.syncthreads()

        stride *= 2

    if tx < n:
        out[tx] = s1[tx]


X = np.array(
    [1, 2, 3, 4, 5],
    dtype=np.float32
)

d_x = cuda.to_device(X)
d_out = cuda.device_array_like(X)

# Single block, power-of-two threads
scan_hillis_steele[1, 8](d_x, d_out)

result = d_out.copy_to_host()
expected = np.cumsum(X)

print("Input:", X)
print("GPU Prefix Sum:", result)
print("CPU Prefix Sum:", expected)

np.testing.assert_allclose(
    result, expected, atol=1e-4
)

print("Task 20: PASS")

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Input: [1. 2. 3. 4. 5.]
GPU Prefix Sum: [ 1.  3.  6. 10. 15.]
CPU Prefix Sum: [ 1.  3.  6. 10. 15.]
Task 20: PASS
